In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

import numpy as np
import matplotlib.pyplot as plt
import pickle
import torch

import sys
sys.path.insert(0, '../../../')

from src.difsched.config import getExpConfig, visualizeExpConfig
from src.difsched.env.Hybrid import createEnv
from src.difsched.utils.EnvInterface import EnvInterface
from src.difsched.evaluation import loadAndEvaluation

In [2]:
datasetFolder = f'../../../data/processed/offline_dataset'


In [4]:
expConfigIdx = 0
expParams = getExpConfig(expConfigIdx)
visualizeExpConfig(expParams)

# ============== Prepare offline dataset ==============
dataset_off = {
    'observations': [],
    'actions': [],
    'rewards': [],
    'next_observations': []
}
for exp_idx in expParams['offline_dataset_idxs']:
    with open(f'{datasetFolder}/subOptimalAgent_envConfig{exp_idx}.pkl', 'rb') as f:
        dataset_expert = pickle.load(f)
    
    dataset_off['observations'].extend(dataset_expert['uRecord'])
    dataset_off['actions'].extend(dataset_expert['actionsRecord'])
    dataset_off['rewards'].extend(dataset_expert['rewardRecord'])
    dataset_off['next_observations'].extend(dataset_expert['uNextRecord'])
    
    print(f"Exp {exp_idx} - Avg. packet loss rate: {np.mean(dataset_expert['rewardRecord'])}")
    print(f"Exp {exp_idx} - length of dataset: {len(dataset_expert['uRecord'])}")

print(f"\nCombined dataset length: {len(dataset_off['observations'])}")
print(f"Combined avg. packet loss rate: {np.mean(dataset_off['rewards'])}")

# ============== Prepare environment ==============

trafficDataParentPath = f'../../../data/processed/traffic'
env = createEnv(expParams, trafficDataParentPath)
env.selectMode(mode="test", type="data")
envInterface = EnvInterface(expParams, discrete_state=False)

modelFolder = f'../../../data/results/dql/config_{expConfigIdx}'
loadAndEvaluation(env, envInterface, dataset_expert, modelFolder, exp_idx_list=[1])


EnvType: HYBRID
N_user: 8
LEN_window: 200
N_aggregation: 4
dataflow: thumb_fr
randomSeed: 999
r_bar: 4
B: 40
sigma_list: [0.7, 0.8, 0.9]
offline_dataset_idxs: [0, 1, 2]
Exp 0 - Avg. packet loss rate: 0.004995622282394194
Exp 0 - length of dataset: 10000
Exp 1 - Avg. packet loss rate: 0.004997266152865283
Exp 1 - length of dataset: 10000
Exp 2 - Avg. packet loss rate: 0.004748857103369525
Exp 2 - length of dataset: 10000

Combined dataset length: 30000
Combined avg. packet loss rate: 0.004913915179543002
Expert's Reward: 0.004748857103369525


reward_diffusionQ1: 0.005124084097027247
best_exp_idx: 1


IndexError: list index out of range